In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
q1 = "can i still join the course after the start date?"

v1 = model.encode(q1)

document = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

dv = model.encode(document)


In [4]:
# find the cosine similarity between the question and the document
v1.dot(dv)

np.float32(0.32332397)

In [5]:
q2 = "how do i install docker on windows?"

v2 = model.encode(q2)
# find the cosine similarity between the question and the document
v2.dot(dv)



np.float32(0.023847342)

In [6]:
from ingest import load_faq_data

documents = load_faq_data()

[{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 471}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 253}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 118}]


In [7]:
texts = [(' ').join([doc['question'], doc['answer']]) for doc in documents]
texts[0]

"How do I submit homework? - Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"

In [ ]:
# embed documents into vector space
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
  batch = texts[i:i + batch_size]
  bactch_vectors = model.encode(batch)
  vectors.extend(bactch_vectors)



  0%|          | 0/28 [00:00<?, ?it/s]

1380

In [46]:
from dataclasses import dataclass
from operator import attrgetter


@dataclass
class Document:
  id: int
  text: str
  sim: float

  # def __lt__(self, other):
  #   return self.sim < other.sim

def search(query: str, vectors: list[float], documents: list[dict]) -> Document:
  v1 = model.encode(query)
  max_sim = max(
      (Document(i, documents[i]['answer'], float(v1.dot(vectors[i])))
       for i in range(len(vectors))),
      key=lambda d: d.sim,
  )
  return max_sim.text

# max_sim = max(
#     (Document(i, documents[i]['answer'], float(v1.dot(vectors[i])))
#      for i in range(len(vectors))),
#     key=lambda d: d.sim,
# )

# max_sim = max(
#     (Document(i, documents[i]['answer'], float(v1.dot(vectors[i])))
#      for i in range(len(vectors))),
#     key=attrgetter('sim'),
# )

search("how do i install docker on windows?", vectors, documents)


'On some versions of Ubuntu, the `snap` command can be used to install Docker.\n\n```bash\nsudo snap install docker\n```'

In [ ]:
import numpy as np

# [
#   [0.1, 0.2, 0.3], # document 1
#   [0.4, 0.5, 0.6], # document 2
#   [0.7, 0.8, 0.9]  # document 3
# ]
# matrix vector multiplication is very fast
matrix_vectors = np.array(vectors)

scores = matrix_vectors.dot(v1)

print(max(scores))


0.7629411


In [59]:
vector_index = np.argmax(scores)
vector_index, scores[vector_index]



(np.int64(860), np.float32(0.7629411))

In [69]:

# select top 5 documents
top_5_indices = np.argsort(scores)[-5:]
top_5_indices, scores[top_5_indices]

(array([ 865, 1262,   29,  473,  860]),
 array([0.5600999 , 0.6536313 , 0.71921337, 0.7579371 , 0.7629411 ],
       dtype=float32))

In [ ]:
# [::-1] is a shortcut to reverse the list
[(scores[idx], documents[idx]) for idx in top_5_indices[::-1]]

# this would also reverse the list
top5indeces = np.argsort(-scores)[:5]
top5indeces, scores[top5indeces]


(array([ 860,  473,   29, 1262,  865]),
 array([0.7629411 , 0.7579371 , 0.71921337, 0.6536313 , 0.5600999 ],
       dtype=float32))

In [ ]:
# vector search with minsearch
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(matrix_vectors, documents)
vindex.search(v1, filter_dict={'course': 'llm-zoomcamp'}, num_results=5, output_ids=True)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  '_id': 1262},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  '_id': 1270},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in 

In [88]:
# implement RAG with vector search
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI
from ingest import load_faq_data, build_index
from typing import TypedDict, Unpack

load_dotenv()
openai_client = OpenAI()

documents = load_faq_data()
index = build_index(documents)

class RAGBaseKwargs(TypedDict, total=False):
    instructions: str
    prompt_template: str
    course: str
    model: str

class VectorSearchRAG(RAGBase):
  def __init__(self, embedder, index, llm_client, **kwargs: Unpack[RAGBaseKwargs]):
    super().__init__(index=index, llm_client=llm_client, **kwargs)
    self.embedder = embedder

  def search(self, query, num_results=5):
    query_vector = self.embedder.encode(query)
    filter_dict = {'course': self.course}

    return self.index.search(
      query_vector,
      filter_dict=filter_dict,
      num_results=num_results
    )

# from sentence_transformers import SentenceTransformer

# model = SentenceTransformer('all-MiniLM-L6-v2')
# ---
# vector search with minsearch
# from minsearch import VectorSearch

# vindex = VectorSearch(keyword_fields=['course'])
# vindex.fit(matrix_vectors, documents)
vector_assistant = VectorSearchRAG(
  embedder=model,
  index=vindex,
  llm_client=openai_client
)

response = vector_assistant.rag('can i still take this course?')





[{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 471}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 253}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 118}]


In [89]:
print(response)

Yes, you can still take the course. If you want a certificate, you need to submit your project while the course is still accepting submissions.
